# AutoResearch on Kaggle

基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：** Settings → Accelerator → 選 GPU T4 x2 或 P100

> Kaggle 提供免費 GPU（T4/P100），此 notebook 會自動偵測 GPU 並套用相容性 patch。

In [ ]:
# 確認 GPU 並偵測架構
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"\n✅ GPU: {name} (compute capability: {cap[0]}.{cap[1]})")
    if cap[0] >= 8:
        print("   Ampere+ → Flash Attention 3 原生支援")
    else:
        print("   Pre-Ampere → 將自動套用 SDPA fallback patch")
else:
    print("❌ 未偵測到 GPU！請到 Settings → Accelerator → 選 GPU")

In [ ]:
# 安裝 uv（Kaggle 環境預裝 pip，但 autoresearch 用 uv）
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
# Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

In [ ]:
# 安裝依賴 + 套用 T4/P100 相容性 patch
!uv sync

# 下載 T4 patch 並套用
!curl -sL https://raw.githubusercontent.com/human530/MD.Piece/main/colab_t4_patch.py -o /tmp/t4_patch.py
!python /tmp/t4_patch.py

In [ ]:
# 資料準備（下載 + tokenizer，約 2-3 分鐘）
!uv run prepare.py --num-shards 10

In [ ]:
# 跑一次訓練（約 5-10 分鐘，視 GPU 而定）
!uv run train.py 2>&1 | tee /tmp/run.log

# 提取結果
import re
log = open("/tmp/run.log").read()
bpb_match = re.findall(r"val_bpb[:\s]+([\d.]+)", log)
loss_match = re.findall(r"train_loss[:\s]+([\d.]+)", log)
step_match = re.findall(r"step[:\s]+(\d+)", log)

val_bpb = float(bpb_match[-1]) if bpb_match else None
train_loss = float(loss_match[-1]) if loss_match else None
steps = int(step_match[-1]) if step_match else None

print(f"\n{'='*40}")
print(f"val_bpb:    {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps:      {steps}")
print(f"{'='*40}")

## 回傳結果到 MD.Piece API

設定你的 MD.Piece 後端 URL。

**方式 1：** 用 ngrok 暴露本機 port 8000

**方式 2：** 部署到公開伺服器

**方式 3：** 留空，結果存在本地 TSV，之後手動上傳

In [ ]:
# ===== 設定區 =====
API_URL = ""  # 填入你的 MD.Piece 後端 URL，例如 "https://xxxx.ngrok.io"
EXPERIMENT_NAME = "baseline-kaggle"
NOTES = "Kaggle GPU baseline run"
# ==================

import requests, json

payload = {
    "name": EXPERIMENT_NAME,
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "notes": NOTES,
    "kept": True,
}

if API_URL:
    try:
        r = requests.post(f"{API_URL}/research/", json=payload, timeout=10)
        if r.ok:
            print(f"✅ 已回傳到 MD.Piece: {r.json()}")
        else:
            print(f"⚠️ API 回傳錯誤 ({r.status_code}): {r.text}")
    except Exception as e:
        print(f"⚠️ 無法連線到 {API_URL}: {e}")
        print(f"📋 手動提交指令：")
        print(f'curl -X POST {API_URL}/research/ -H "Content-Type: application/json" -d \'{json.dumps(payload)}\'')
else:
    print("ℹ️ 未設定 API_URL，結果僅存在本地")
    print(f"📋 結果：{json.dumps(payload, indent=2)}")

## 自動化實驗循環

下載 `autoresearch_runner.py` 自動跑多輪實驗（假設 → 修改 → 訓練 → 評估 → keep/revert）。

In [ ]:
# 下載自動化 runner 並執行完整實驗循環
!pip install requests -q

# 下載 runner（嘗試 main branch，失敗則用當前 branch）
import urllib.request, os

runner_urls = [
    "https://raw.githubusercontent.com/human530/MD.Piece/main/autoresearch_runner.py",
    "https://raw.githubusercontent.com/human530/MD.Piece/claude/check-autoresearch-OlWKc/autoresearch_runner.py",
]

downloaded = False
for url in runner_urls:
    try:
        urllib.request.urlretrieve(url, "autoresearch_runner.py")
        if os.path.getsize("autoresearch_runner.py") > 100:
            print(f"✅ Downloaded runner from: {url}")
            downloaded = True
            break
    except Exception as e:
        print(f"⚠️ Failed to download from {url}: {e}")

if not downloaded:
    raise RuntimeError("❌ Could not download autoresearch_runner.py from any source")

# ===== 設定區 =====
API_URL = ""  # 填入你的 MD.Piece 後端 URL（留空則不回傳）
ROUNDS = 5
# ==================

# 初始化 git（runner 需要 git commit/revert）
!git config user.email "kaggle@autoresearch" && git config user.name "autoresearch"
!git init . 2>/dev/null; git add -A; git commit -m "initial" --allow-empty 2>/dev/null

# 開始自動實驗循環
api_flag = f"--api-url {API_URL}" if API_URL else ""
!python autoresearch_runner.py --rounds {ROUNDS} {api_flag}

In [ ]:
# 查看結果摘要
import pandas as pd
from pathlib import Path

results_file = Path("results.tsv")
if not results_file.exists():
    # 可能在 autoresearch/ 子目錄
    alt = Path("autoresearch/results.tsv")
    if alt.exists():
        results_file = alt

if results_file.exists() and results_file.stat().st_size > 0:
    df = pd.read_csv(results_file, sep="\t")
    print(df.to_string(index=False))
    if "val_bpb" in df.columns and df["val_bpb"].notna().any():
        best_idx = df["val_bpb"].idxmin()
        print(f"\n🏆 Best: {df.loc[best_idx, 'name']} — val_bpb: {df['val_bpb'].min()}")
    else:
        print("\n⚠️ No valid val_bpb values found")
else:
    print("❌ results.tsv not found — the runner may have failed.")
    print("   Check the output of the previous cell for errors.")
    print("   Common causes:")
    print("   1. autoresearch_runner.py download failed")
    print("   2. Baseline training failed (GPU not enabled?)")
    print("   3. train.py not found (git clone failed)")

In [ ]:
# 下載 results.tsv（Kaggle 可直接從 Output 下載）
from pathlib import Path
from IPython.display import FileLink, display

for p in [Path("results.tsv"), Path("autoresearch/results.tsv")]:
    if p.exists():
        display(FileLink(str(p)))
        break
else:
    print("⚠️ results.tsv not found — nothing to download")